# Plasma-Cell Spatial Niche Analysis

This notebook focuses the spatial analysis on `Plasma_cell_like` cells. Plasma cells are central to the template study because multiple myeloma is a plasma-cell malignancy and the paper emphasizes plasma-cell neighborhoods, immune context, and bone-associated spatial organization.

This analysis extracts all abundance-normalized spatial enrichment rows where one member of the phenotype pair is `Plasma_cell_like`. The result summarizes which approximate phenotypes are enriched or depleted as plasma-cell-like neighbors.

Interpretation boundary:

- Phenotypes are rule-based approximations.
- Spatial enrichment reflects adjacency relative to phenotype abundance.
- This analysis is a focused exploratory summary, not a formal reproduction of the paper's CellCharter neighborhood analysis or survival model.


## Step 0: Configure Workflow Paths

This setup cell defines the input and output paths for plasma-cell niche analysis.

Inputs:

```text
results/15_phenotype_composition_by_category.csv
results/20_spatial_phenotype_enrichment_by_image.csv
results/21_spatial_phenotype_enrichment_by_category.csv
```

Outputs:

```text
results/22_plasma_cell_niche_by_image.csv
results/23_plasma_cell_niche_by_category.csv
results/24_plasma_cell_niche_summary.csv
figures/08_plasma_cell_neighbor_enrichment.png
```


In [ ]:
from pathlib import Path
import csv
import subprocess

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

RESULTS_DIR = WORKFLOW_DIR / 'results'
FIGURES_DIR = WORKFLOW_DIR / 'figures'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

print('Workflow directory:', WORKFLOW_DIR)
print('Category composition exists:', (RESULTS_DIR / '15_phenotype_composition_by_category.csv').exists())
print('Image enrichment exists:', (RESULTS_DIR / '20_spatial_phenotype_enrichment_by_image.csv').exists())
print('Category enrichment exists:', (RESULTS_DIR / '21_spatial_phenotype_enrichment_by_category.csv').exists())
print('Plasma niche script exists:', (SCRIPTS_DIR / '13_plasma_cell_niche_analysis.py').exists())


## Step 1: Understand the Plasma-Cell Niche Summary

The input enrichment tables contain many phenotype pairs. This notebook filters those tables to retain only pairs involving `Plasma_cell_like`.

Example retained pairs:

```text
Plasma_cell_like -- CD4_T_cell_like
Plasma_cell_like -- CD8_T_cell_like
Plasma_cell_like -- Macrophage_monocyte_like
Plasma_cell_like -- Myeloid_cell_like
Plasma_cell_like -- Unknown
Plasma_cell_like -- Plasma_cell_like
```

For each retained pair, the key metric is `log2_enrichment`:

```text
positive: more adjacent than expected from abundance
zero: approximately expected from abundance
negative: less adjacent than expected from abundance
```

The summary table also reports plasma-cell-like abundance per category so spatial enrichment can be interpreted together with phenotype composition.


## Step 2: Run Plasma-Cell Niche Analysis

The script performs the following operations:

```text
1. Read phenotype composition by category.
2. Read spatial enrichment by image and by category.
3. Keep only rows involving Plasma_cell_like.
4. Convert the paired phenotype into a neighbor_phenotype column.
5. Write plasma-cell niche summaries by image and category.
6. Summarize top enriched and depleted plasma-cell-like neighbors by category using a minimum edge-count threshold for the top-neighbor summary.
7. Save a heatmap of plasma-cell-like neighbor enrichment.
```


In [ ]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '13_plasma_cell_niche_analysis.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--composition-category', 'results/15_phenotype_composition_by_category.csv',
    '--enrichment-image', 'results/20_spatial_phenotype_enrichment_by_image.csv',
    '--enrichment-category', 'results/21_spatial_phenotype_enrichment_by_category.csv',
    '--min-summary-edges', '25',
    '--output-image', 'results/22_plasma_cell_niche_by_image.csv',
    '--output-category', 'results/23_plasma_cell_niche_by_category.csv',
    '--output-summary', 'results/24_plasma_cell_niche_summary.csv',
    '--figure', 'figures/08_plasma_cell_neighbor_enrichment.png',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)


## Step 3: Review Category-Level Plasma-Cell Niche Summary

The summary table gives one row per category. It reports plasma-cell-like abundance and the strongest enriched/depleted plasma-cell-like neighbor phenotypes after applying the minimum edge-count threshold for top-neighbor selection.

Columns:

```text
category
plasma_cell_count
plasma_cell_fraction
plasma_neighbor_pairs
minimum_edges_for_top_neighbor_summary
plasma_neighbor_pairs_passing_minimum_edges
enriched_pairs_log2_gt_0
depleted_pairs_log2_lt_0
top_enriched_neighbor
top_enriched_log2
top_enriched_edges
top_depleted_neighbor
top_depleted_log2
top_depleted_edges
```

This table is designed for final interpretation because it identifies the dominant plasma-cell-like spatial context in each category.


In [ ]:
summary_path = RESULTS_DIR / '24_plasma_cell_niche_summary.csv'
with summary_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        print(row)


## Step 4: Review Plasma-Cell Neighbor Enrichment by Category

The category-level table reports all plasma-cell-like neighbor phenotype pairs. This table is useful for detailed inspection beyond the top enriched and depleted neighbors.

Rows with high positive `log2_enrichment` indicate neighbor phenotypes observed around plasma-cell-like cells more often than expected from abundance. Rows with negative `log2_enrichment` indicate relative depletion.


In [ ]:
category_path = RESULTS_DIR / '23_plasma_cell_niche_by_category.csv'
with category_path.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

for row in rows[:40]:
    print(row)


## Step 5: Review Plasma-Cell Niche Figure

The figure summarizes `Plasma_cell_like` neighbor enrichment by category. Each column is a neighbor phenotype and each row is a category.

Interpretation guidance:

- Positive values indicate enriched plasma-cell-like adjacency.
- Negative values indicate depleted plasma-cell-like adjacency.
- Same-phenotype enrichment reflects plasma-cell-like clustering.
- High enrichment with low edge counts should be interpreted cautiously.


In [ ]:
figure = FIGURES_DIR / '08_plasma_cell_neighbor_enrichment.png'
print('Figure exists:', figure.exists())
print('Figure size_bytes:', figure.stat().st_size if figure.exists() else 0)
